In [9]:
import random

vocabulary = {
    "水果": {
        "apple": "蘋果",
        "banana": "香蕉",
    },
    "動物": {
        "cat": "貓咪",
    },
}

quiz_records = {"correct": [], "incorrect": []}


def all_words():
    """回傳所有單字，格式為 [(英文, 中文, 分類), ...]。"""
    return [
        (english, chinese, category)
        for category, words in vocabulary.items()
        for english, chinese in words.items()
    ]


def show_summary():
    """以文字格式顯示單字總數、分類與答題紀錄。"""
    words = all_words()
    print("=" * 36)
    print("             VocaBuddy 單字總覽")
    print("=" * 36)
    print(f"單字總數：{len(words)}")
    print("\n單字分類：")
    for category, category_words in vocabulary.items():
        print(f"  {category}（{len(category_words)} 個）：")
        print("    " + ", ".join(category_words.keys()))
    print("\n答題紀錄：")
    print(f"  答對：{len(quiz_records['correct'])} 題")
    print(f"  答錯：{len(quiz_records['incorrect'])} 題")
    print("=" * 36)


def run_quiz(question_count=3, answer_provider=input, category=None):
    """進行簡單中文翻英文測驗。"""
    candidates = [word for word in all_words() if category is None or word[2] == category]
    if not candidates:
        print("找不到可測驗的單字。")
        return
    questions = random.sample(candidates, min(question_count, len(candidates)))
    correct_count = 0
    for number, (english, chinese, word_category) in enumerate(questions, start=1):
        answer = answer_provider(f"第 {number} 題 [{word_category}] {chinese} 的英文是：").strip().lower()
        if answer == english.lower():
            print("答對！")
            quiz_records["correct"].append(english)
            correct_count += 1
        else:
            print(f"答錯，正確答案是：{english}")
            quiz_records["incorrect"].append({"question": chinese, "answer": answer, "correct": english})
    print(f"本次成績：{correct_count}/{len(questions)} 題答對")


def add_word(english, chinese, category):
    """新增單字到指定分類。"""
    vocabulary.setdefault(category, {})[english.strip().lower()] = chinese.strip()


In [10]:
import json
from IPython.display import HTML, Javascript, clear_output, display

words_for_ui = [
    {"english": english, "chinese": chinese, "category": category}
    for category, words in vocabulary.items()
    for english, chinese in words.items()
]
words_json = json.dumps(words_for_ui, ensure_ascii=False)

html_ui = """
<style>
#vocabuddy-root { max-width: 760px; margin: 0 auto; padding: 14px; color: #173b50; font: 16px/1.5 "Trebuchet MS", "Segoe UI", sans-serif; }
#vocabuddy-root *, #vocabuddy-root *::before, #vocabuddy-root *::after { box-sizing: border-box; }
.vb-hero { padding: 24px; border: 1px solid #c8e5df; border-radius: 16px; background: linear-gradient(135deg, #e8f7f3, #fff8ed); }
.vb-hero h1 { margin: 0 0 4px; font-size: 32px; }
.vb-hero p, .vb-note { margin: 0; color: #617984; }
.vb-stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin: 14px 0; }
.vb-stat { padding: 14px; border: 1px solid #d9e7e7; border-radius: 12px; background: #fff; }
.vb-stat strong { display: block; color: #087f73; font-size: 25px; }
.vb-stat span { color: #617984; font-size: 13px; }
.vb-panel { padding: 18px; border: 1px solid #d9e7e7; border-radius: 14px; background: #fff; }
.vb-tabs, .vb-actions, .vb-row { display: flex; flex-wrap: wrap; gap: 8px; }
.vb-tabs { margin-bottom: 16px; }
.vb-tabs button, .vb-btn { min-height: 44px; padding: 9px 14px; border: 1px solid #d9e7e7; border-radius: 8px; background: #fff; color: #173b50; font: inherit; font-weight: 700; cursor: pointer; }
.vb-tabs button.active, .vb-btn.primary { border-color: #087f73; background: #087f73; color: #fff; }
.vb-btn:disabled { cursor: not-allowed; opacity: .45; }
.vb-card { display: grid; place-items: center; min-height: 190px; margin-top: 14px; border-radius: 14px; background: #173b50; color: #fff; text-align: center; cursor: pointer; }
.vb-card small { display: block; margin-bottom: 8px; color: #bfe5de; }
.vb-card strong { font-size: 44px; }
.vb-row { align-items: end; margin-bottom: 14px; }
.vb-field { display: grid; gap: 4px; color: #617984; font-size: 13px; font-weight: 700; }
.vb-select, .vb-input { min-height: 44px; padding: 9px; border: 1px solid #d9e7e7; border-radius: 8px; background: #fff; color: #173b50; font: inherit; }
.vb-input { flex: 1; min-width: 180px; }
.vb-question { padding: 20px; border-radius: 12px; background: #173b50; color: #fff; }
.vb-question small { display: block; color: #bfe5de; }
.vb-question strong { font-size: 28px; }
.vb-feedback { min-height: 24px; margin-top: 10px; padding: 9px; border-radius: 8px; color: #617984; }
.vb-good { background: #e4f6eb; color: #17633d; }
.vb-bad { background: #fff0ed; color: #a34437; }
.vb-progress { height: 6px; margin: 12px 0; border-radius: 9px; background: #e4eeee; overflow: hidden; }
.vb-progress span { display: block; height: 100%; background: #087f73; }
.vb-mode { display: none; }
.vb-mode.active { display: block; }
@media (max-width: 560px) { .vb-stats { grid-template-columns: 1fr; } .vb-card strong { font-size: 34px; } .vb-row { flex-direction: column; align-items: stretch; } }
</style>
<div id="vocabuddy-root">
  <header class="vb-hero"><h1>VocaBuddy</h1><p>翻卡學單字，回答小測驗。</p></header>
  <div class="vb-stats"><div class="vb-stat"><strong id="total">0</strong><span>單字總數</span></div><div class="vb-stat"><strong id="correct">0</strong><span>答對題數</span></div><div class="vb-stat"><strong id="accuracy">0%</strong><span>目前正確率</span></div></div>
  <main class="vb-panel">
    <div class="vb-tabs"><button class="active" data-mode="cards" type="button">單字卡</button><button data-mode="quiz" type="button">單字測驗</button></div>
    <section id="cards" class="vb-mode active"><div class="vb-row"><label class="vb-field">分類<select id="cardCategory" class="vb-select"></select></label><span id="position" class="vb-note"></span></div><div id="card" class="vb-card" tabindex="0" role="button"><div><small id="cardLabel"></small><strong id="cardWord"></strong></div></div><p class="vb-note" style="text-align:center;margin:10px 0">點擊卡片或按 Enter 翻面</p><div class="vb-actions"><button class="vb-btn" id="prev" type="button">上一張</button><button class="vb-btn primary" id="next" type="button">下一張</button><button class="vb-btn" id="random" type="button">隨機單字</button></div></section>
    <section id="quiz" class="vb-mode"><div class="vb-row"><label class="vb-field">分類<select id="quizCategory" class="vb-select"></select></label><label class="vb-field">題數<select id="quizLength" class="vb-select"><option>3</option><option>5</option><option>10</option></select></label><button class="vb-btn primary" id="start" type="button">開始測驗</button></div><div id="quizBody" hidden><div id="question" class="vb-question"></div><div class="vb-progress"><span id="progress"></span></div><div class="vb-row"><input id="answer" class="vb-input" placeholder="輸入英文答案"><button class="vb-btn primary" id="submit" type="button">確認答案</button><button class="vb-btn" id="nextQuiz" type="button" disabled>下一題</button></div><div id="feedback" class="vb-feedback">輸入答案後按確認。</div></div></section>
  </main>
</div>
"""

js_ui = """
(function bindVocaBuddy() {
  const root = document.getElementById("vocabuddy-root");
  if (!root) { setTimeout(bindVocaBuddy, 50); return; }
  const WORDS = __WORDS_JSON__;
  const records = { correct: 0, incorrect: 0 };
  let cards = [...WORDS], cardIndex = 0, flipped = false;
  let quiz = [], quizIndex = 0, sessionCorrect = 0, answered = false;
  const $ = (id) => root.querySelector(`#${id}`);
  const categories = ["全部分類", ...new Set(WORDS.map((word) => word.category))];
  const options = categories.map((category) => `<option>${category}</option>`).join("");
  $("cardCategory").innerHTML = options; $("quizCategory").innerHTML = options;
  $("total").textContent = WORDS.length;
  function stats() { const total = records.correct + records.incorrect; $("correct").textContent = records.correct; $("accuracy").textContent = `${total ? Math.round(records.correct / total * 100) : 0}%`; }
  function renderCard() { const word = cards[cardIndex]; if (!word) return; $("cardLabel").textContent = flipped ? "中文意思" : word.category; $("cardWord").textContent = flipped ? word.chinese : word.english; $("position").textContent = `${cardIndex + 1} / ${cards.length}`; }
  function changeCard(step) { cardIndex = (cardIndex + step + cards.length) % cards.length; flipped = false; renderCard(); }
  function startQuiz() { const category = $("quizCategory").value; const pool = WORDS.filter((word) => category === "全部分類" || word.category === category); quiz = [...pool].sort(() => Math.random() - .5).slice(0, Math.min(Number($("quizLength").value), pool.length)); quizIndex = 0; sessionCorrect = 0; $("quizBody").hidden = false; showQuestion(); }
  function showQuestion() { if (quizIndex >= quiz.length) { $("question").innerHTML = `<small>測驗完成</small><strong>${sessionCorrect} / ${quiz.length} 題答對</strong>`; $("feedback").className = "vb-feedback vb-good"; $("feedback").textContent = "完成測驗！"; $("submit").disabled = true; $("nextQuiz").disabled = true; $("progress").style.width = "100%"; return; } const word = quiz[quizIndex]; answered = false; $("question").innerHTML = `<small>第 ${quizIndex + 1} / ${quiz.length} 題 · ${word.category}</small><strong>${word.chinese}</strong>`; $("progress").style.width = `${quizIndex / quiz.length * 100}%`; $("answer").value = ""; $("answer").disabled = false; $("submit").disabled = false; $("nextQuiz").disabled = true; $("feedback").className = "vb-feedback"; $("feedback").textContent = "輸入答案後按確認。"; }
  function submitAnswer() { if (answered || quizIndex >= quiz.length) return; answered = true; const word = quiz[quizIndex]; const answer = $("answer").value.trim().toLowerCase(); $("answer").disabled = true; $("submit").disabled = true; $("nextQuiz").disabled = false; if (answer === word.english.toLowerCase()) { sessionCorrect++; records.correct++; $("feedback").className = "vb-feedback vb-good"; $("feedback").textContent = "答對了！"; } else { records.incorrect++; $("feedback").className = "vb-feedback vb-bad"; $("feedback").innerHTML = `正確答案是 <b>${word.english}</b>`; } stats(); }
  root.querySelectorAll("[data-mode]").forEach((button) => button.onclick = () => { root.querySelectorAll("[data-mode]").forEach((item) => item.classList.remove("active")); button.classList.add("active"); $("cards").classList.toggle("active", button.dataset.mode === "cards"); $("quiz").classList.toggle("active", button.dataset.mode === "quiz"); });
  $("card").onclick = () => { flipped = !flipped; renderCard(); }; $("card").onkeydown = (event) => { if (event.key === "Enter" || event.key === " ") { event.preventDefault(); flipped = !flipped; renderCard(); } };
  $("prev").onclick = () => changeCard(-1); $("next").onclick = () => changeCard(1); $("random").onclick = () => { cardIndex = Math.floor(Math.random() * cards.length); flipped = false; renderCard(); };
  $("cardCategory").onchange = (event) => { cards = event.target.value === "全部分類" ? [...WORDS] : WORDS.filter((word) => word.category === event.target.value); cardIndex = 0; flipped = false; renderCard(); };
  $("start").onclick = startQuiz; $("submit").onclick = submitAnswer; $("nextQuiz").onclick = () => { quizIndex++; showQuestion(); }; $("answer").onkeydown = (event) => { if (event.key === "Enter") submitAnswer(); };
  renderCard(); stats();
})();
""".replace("__WORDS_JSON__", words_json)

clear_output(wait=True)
display(HTML(html_ui))
display(Javascript(js_ui))

<IPython.core.display.Javascript object>